RAG Pipeline - Data Ingestion to vector DB pipeline

In [19]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [20]:
### Read all the pdf's insider the directory
def process_all_pdfs(pdf_directory):
    """process all pdf files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all pdf recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source"] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file.name} successfully.")

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")
    print(f"\nFinished processing. Total documents loaded: {len(all_documents)}")
    return all_documents

#process all PDFs on the data directory

all_pdf_documents = process_all_pdfs("c:/Code/RAG/data/pdf")

Ignoring wrong pointing object 7 0 (offset 0)
Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 44 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)
Ignoring wrong pointing object 55 0 (offset 0)
Ignoring wrong pointing object 58 0 (offset 0)
Ignoring wrong pointing object 60 0 (offset 0)
Ignoring wrong pointing object 62 0 (offset 0)
Ignoring wrong 

found 3 PDF files to process.

Processing: LLM24aug.pdf
Loaded 72 pages from LLM24aug.pdf successfully.

Processing: Mayo Oshin, Nuno Campos - Learning LangChain_ Building AI and LLM Applications with LangChain and LangGraph (2025, O'Reilly Media) - libgen.li.pdf
Loaded 297 pages from Mayo Oshin, Nuno Campos - Learning LangChain_ Building AI and LLM Applications with LangChain and LangGraph (2025, O'Reilly Media) - libgen.li.pdf successfully.

Processing: sample-pdf-a4-size-65kb.pdf
Loaded 5 pages from sample-pdf-a4-size-65kb.pdf successfully.

Finished processing. Total documents loaded: 374


### Embedding and VectorStore DB

In [23]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingManager:
    """Handles document embedding generation using sentence transformers"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initilize the embedding manager

        Args:
            model_name (str): HuggingFace sentence transformer model name to use for embedding generation
        """

        self.model_name = model_name
        self.model = None
        self.load_model()
    
    def _load_model(self):
        """load sentence transformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully. Embedding dimension:", self.model.get_sentence_embedding_dimension())
        except Exception as e:
            print(f"Error loading embedding model: {e}")
    
    def generate_embeddings(self, text: List[str]) -> np.ndarray:
        """
        generate embeddings for a list of text documents

        Args:
            text: list of strings to embed
        Returns:
            numpy array of shape (len(text), embedding_dim) containing the generated embeddings
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        
        print(f"Generating embeddings for {len(text)} documents...")
        embedding= self.model.encode(text, show_progress_bar=True)
        print(f"generated embeddings with shape: {embedding.shape}")
        return embedding

#initialize the embedding manager
embedding_manager = EmbeddingManager() 
embedding_manager